# Model comparison - Random Forest vs XGBoost

**Purpose (Day 16):** one side-by-side table and the confusion matrices for the paper. Run `ml/train_rf.py` and `ml/train_xgb.py` first (same `--seed`), and optionally `ml/run_seeds.py`.

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# find the repo root (the folder that contains client/) no matter where Jupyter was started
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "client").is_dir())
for sub in ("client", "ml", "capture"):
    sys.path.insert(0, str(ROOT / sub))
RESULTS = ROOT / "results"
print("repo root:", ROOT)

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 200)
SEED = 42
runs = {"Random Forest": RESULTS / f"ml_rf_seed{SEED}.json", "XGBoost": RESULTS / f"ml_xgb_seed{SEED}.json"}
M = {name: json.loads(p.read_text()) for name, p in runs.items() if p.exists()}
print("found:", list(M))
cols = ["accuracy", "precision", "recall", "f1", "roc_auc", "false_positive_rate", "fit_seconds"]
table = pd.DataFrame({k: {c: v.get(c) for c in cols} for k, v in M.items()}).T
table.round(4)

## Confusion matrices
Rows = truth, columns = prediction. The off-diagonal cells are the mistakes: **FP** = benign flagged malicious (false alarm), **FN** = malicious missed.

In [ ]:
fig, axes = plt.subplots(1, len(M), figsize=(4.6 * len(M), 4), squeeze=False)
for ax, (name, m) in zip(axes[0], M.items()):
    cm = np.array([[m["tn"], m["fp"]], [m["fn"], m["tp"]]])
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]}", ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=14)
    ax.set_xticks([0, 1], ["benign", "malicious"]); ax.set_yticks([0, 1], ["benign", "malicious"])
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(name)
fig.tight_layout(); fig.savefig(RESULTS / "fig_confusion_matrices.png", dpi=300); plt.show()

## Which features did the models use?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, (tag, name) in zip(axes, (("rf", "Random Forest"), ("xgb", "XGBoost"))):
    p = RESULTS / f"ml_{tag}_importance_seed{SEED}.csv"
    if p.exists():
        d = pd.read_csv(p).head(10).iloc[::-1]
        ax.barh(d.feature, d.importance); ax.set_title(f"{name}: top 10 features")
fig.tight_layout(); plt.show()

## Mean ± std over seeds (from `ml/run_seeds.py`)

In [ ]:
p = RESULTS / "ml_multiseed_summary.csv"
pd.read_csv(p).round(4) if p.exists() else print("run: python ml/run_seeds.py  to create", p.name)